# Exponential decay: unbinned and binned likelihood

The two original decay examples are combined here so that the same events, observable range, and exponential model can be fitted in two ways.

For $0\le t\le 5$ s, <code>RooExponential</code> represents
$$
f(t\mid a)=\frac{e^{at}}{\int_0^5 e^{au}\,du},\qquad a<0.
$$
The physical lifetime is $T=-1/a$. The finite range is included in the normalization.

## Unbinned likelihood

For measured event times $\{t_i\}$,
$$
\mathcal L_{\mathrm{unbinned}}(a)=\prod_{i=1}^{N_{\mathrm{obs}}} f(t_i\mid a).
$$
The fit uses each event time directly. The bins visible in the plot are only a display choice.

## Binned likelihood

With fixed bin edges, let $n_j$ be the observed count and $\mu_j$ the model expectation in bin $j$. The binned Poisson likelihood is
$$
\mathcal L_{\mathrm{binned}}=\prod_j\frac{\mu_j^{n_j}e^{-\mu_j}}{n_j!}.
$$
RooFit evaluates this likelihood when the data are stored in a <code>RooDataHist</code>.

## Extended likelihood

Both fits below use an extended likelihood, so the expected yield $N$ is fitted together with the decay slope. The same generated <code>RooDataSet</code> is used for both fits; only the representation supplied to <code>fitTo</code> changes. Comparing the fitted lifetime and yield shows the information change introduced by fixed bins without confusing it with a different random sample.



<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code:</span>
  <button type="button" data-code-language="python" aria-pressed="true">PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:flex; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  selectLanguage("python");
});
</script>


<div class="pyroot-code-marker"></div>

```python
import ROOT

ROOT.RooRandom.randomGenerator().SetSeed(42)
expected_events = 100
display_bins = 50

# Generate one event-by-event sample.
t = ROOT.RooRealVar("t", "time", 0, 5, "s")
t.setBins(display_bins)
slope_gen = ROOT.RooRealVar("slope_gen", "decay slope", -1.0)
pdf_gen = ROOT.RooExponential("pdf_gen", "decay PDF", t, slope_gen)
yield_gen = ROOT.RooRealVar("yield_gen", "expected events", expected_events)
model_gen = ROOT.RooExtendPdf("model_gen", "extended decay model", pdf_gen, yield_gen)
data = model_gen.generate(ROOT.RooArgSet(t), ROOT.RooFit.Extended(True))
data_hist = ROOT.RooDataHist("data_hist", "binned data", ROOT.RooArgSet(t), data)

# Unbinned extended-likelihood fit
slope_u = ROOT.RooRealVar("slope_u", "decay slope", -0.8, -5.0, -0.05)
yield_u = ROOT.RooRealVar("yield_u", "expected events", 90, 0, 300)
pdf_u = ROOT.RooExponential("pdf_u", "decay PDF", t, slope_u)
model_u = ROOT.RooExtendPdf("model_u", "unbinned model", pdf_u, yield_u)
result_u = model_u.fitTo(
    data, ROOT.RooFit.Save(True), ROOT.RooFit.Extended(True), ROOT.RooFit.PrintLevel(-1)
)

# Binned extended-likelihood fit to the same events
slope_b = ROOT.RooRealVar("slope_b", "decay slope", -0.8, -5.0, -0.05)
yield_b = ROOT.RooRealVar("yield_b", "expected events", 90, 0, 300)
pdf_b = ROOT.RooExponential("pdf_b", "decay PDF", t, slope_b)
model_b = ROOT.RooExtendPdf("model_b", "binned model", pdf_b, yield_b)
result_b = model_b.fitTo(
    data_hist, ROOT.RooFit.Save(True), ROOT.RooFit.Extended(True), ROOT.RooFit.PrintLevel(-1)
)

def print_result(label, result, slope, expected_yield):
    lifetime = -1.0 / slope.getVal()
    lifetime_error = slope.getError() / slope.getVal() ** 2
    print(
        f"{label}: status={result.status()}, covQual={result.covQual()}, "
        f"lifetime={lifetime:.4f} +/- {lifetime_error:.4f} s, "
        f"yield={expected_yield.getVal():.2f} +/- {expected_yield.getError():.2f}"
    )

print(f"observed events = {data.numEntries()}")
print_result("unbinned", result_u, slope_u, yield_u)
print_result("binned", result_b, slope_b, yield_b)

# The top row shows the fits; the bottom row shows pulls from the displayed bins.
frame_u = t.frame(ROOT.RooFit.Title("Unbinned likelihood"))
data.plotOn(frame_u, ROOT.RooFit.Binning(display_bins),
            ROOT.RooFit.DataError(ROOT.RooAbsData.Poisson), ROOT.RooFit.Name("data_u"))
model_u.plotOn(frame_u, ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("curve_u"))
pull_u = frame_u.pullHist("data_u", "curve_u")
pull_frame_u = t.frame(ROOT.RooFit.Title("Unbinned display pull"))
pull_frame_u.addPlotable(pull_u, "P")
pull_frame_u.SetMinimum(-5); pull_frame_u.SetMaximum(5)

frame_b = t.frame(ROOT.RooFit.Title("Binned likelihood"))
data_hist.plotOn(frame_b, ROOT.RooFit.DataError(ROOT.RooAbsData.Poisson),
                 ROOT.RooFit.Name("data_b"))
model_b.plotOn(frame_b, ROOT.RooFit.LineColor(ROOT.kBlue + 1), ROOT.RooFit.Name("curve_b"))
pull_b = frame_b.pullHist("data_b", "curve_b")
pull_frame_b = t.frame(ROOT.RooFit.Title("Binned pull"))
pull_frame_b.addPlotable(pull_b, "P")
pull_frame_b.SetMinimum(-5); pull_frame_b.SetMaximum(5)

canvas = ROOT.TCanvas("c_decay_compare_py", "Decay likelihood comparison", 1000, 700)
canvas.Divide(2, 2)
canvas.cd(1); frame_u.Draw()
canvas.cd(2); frame_b.Draw()
canvas.cd(3); pull_frame_u.Draw()
canvas.cd(4); pull_frame_b.Draw()
canvas.Draw()
```


In [1]:
#include "RooRealVar.h"
#include "RooExponential.h"
#include "RooExtendPdf.h"
#include "RooDataSet.h"
#include "RooDataHist.h"
#include "RooPlot.h"
#include "RooFitResult.h"
#include "RooRandom.h"
#include "TCanvas.h"
#include <cmath>
#include <iostream>

using namespace RooFit;

RooRandom::randomGenerator()->SetSeed(42);
const int expectedEvents = 100;
const int displayBins = 50;

// Generate one event-by-event sample.
RooRealVar t("t", "time", 0, 5, "s");
t.setBins(displayBins);
RooRealVar slopeGen("slopeGen", "decay slope", -1.0);
RooExponential pdfGen("pdfGen", "decay PDF", t, slopeGen);
RooRealVar yieldGen("yieldGen", "expected events", expectedEvents);
RooExtendPdf modelGen("modelGen", "extended decay model", pdfGen, yieldGen);
auto dataSet = modelGen.generate(t, Extended(true));
RooDataHist dataHist("dataHist", "binned data", t, *dataSet);

// Unbinned extended-likelihood fit
RooRealVar slopeU("slopeU", "decay slope", -0.8, -5.0, -0.05);
RooRealVar yieldU("yieldU", "expected events", 90, 0, 300);
RooExponential pdfU("pdfU", "decay PDF", t, slopeU);
RooExtendPdf modelU("modelU", "unbinned model", pdfU, yieldU);
auto resultU = modelU.fitTo(*dataSet, Save(true), Extended(true), PrintLevel(-1));

// Binned extended-likelihood fit to the same events
RooRealVar slopeB("slopeB", "decay slope", -0.8, -5.0, -0.05);
RooRealVar yieldB("yieldB", "expected events", 90, 0, 300);
RooExponential pdfB("pdfB", "decay PDF", t, slopeB);
RooExtendPdf modelB("modelB", "binned model", pdfB, yieldB);
auto resultB = modelB.fitTo(dataHist, Save(true), Extended(true), PrintLevel(-1));

auto printResult = [](const char* label, const RooFitResult& result,
                      const RooRealVar& slope, const RooRealVar& yield) {
    double lifetime = -1.0 / slope.getVal();
    double lifetimeError = slope.getError() / (slope.getVal() * slope.getVal());
    std::cout << label << ": status=" << result.status()
              << ", covQual=" << result.covQual()
              << ", lifetime=" << lifetime << " +/- " << lifetimeError
              << " s, yield=" << yield.getVal() << " +/- " << yield.getError() << "\n";
};

std::cout << "observed events = " << dataSet->numEntries() << "\n";
printResult("unbinned", *resultU, slopeU, yieldU);
printResult("binned", *resultB, slopeB, yieldB);

// The top row shows the fits; the bottom row shows pulls from the displayed bins.
auto frameU = t.frame(Title("Unbinned likelihood"));
dataSet->plotOn(frameU, Binning(displayBins), DataError(RooAbsData::Poisson), Name("dataU"));
modelU.plotOn(frameU, LineColor(kRed), Name("curveU"));
auto pullU = frameU->pullHist("dataU", "curveU");
auto pullFrameU = t.frame(Title("Unbinned display pull"));
pullFrameU->addPlotable(pullU, "P");
pullFrameU->SetMinimum(-5); pullFrameU->SetMaximum(5);

auto frameB = t.frame(Title("Binned likelihood"));
dataHist.plotOn(frameB, DataError(RooAbsData::Poisson), Name("dataB"));
modelB.plotOn(frameB, LineColor(kBlue + 1), Name("curveB"));
auto pullB = frameB->pullHist("dataB", "curveB");
auto pullFrameB = t.frame(Title("Binned pull"));
pullFrameB->addPlotable(pullB, "P");
pullFrameB->SetMinimum(-5); pullFrameB->SetMaximum(5);

auto canvas = new TCanvas("c_decay_compare", "Decay likelihood comparison", 1000, 700);
canvas->Divide(2, 2);
canvas->cd(1); frameU->Draw();
canvas->cd(2); frameB->Draw();
canvas->cd(3); pullFrameU->Draw();
canvas->cd(4); pullFrameB->Draw();
canvas->Draw();

[#1] INFO:Fitting -- RooAbsPdf::fitTo(modelU) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 2.76058 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_modelU_modelGenData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Fitting -- RooAbsPdf::fitTo(modelB) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 141.125 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_modelB_dataHist) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
observed events = 106
unbinned: status=0, covQual=3, lifetime=0.832713 +/- 0.0847132 s, yield=105